# Run A: baseline evaluation

Setup tasks 2, 8 and 9: load Qwen2.5-Coder-0.5B-Instruct, check it writes code,
then score it on the MBPP test (500) and heldout (60) splits.

Use a GPU runtime (Runtime → Change runtime type → T4 GPU). Results are saved
to Google Drive, so if Colab disconnects, rerun the cells: finished parts are skipped.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUTPUT_DIR = '/content/drive/MyDrive/self-improving-slm'   # shared results folder

In [ ]:
import os
%cd /content
if os.path.exists('Self-Improving-SLM'):
    !cd Self-Improving-SLM && git pull
else:
    !git clone https://github.com/DhariniVenki/Self-Improving-SLM.git
%cd /content/Self-Improving-SLM

vLLM is roughly 10x faster than Hugging Face generation, which matters for pass@8
(4,000 samples). If installing it causes errors, set `USE_VLLM = False`; everything
still works, just slower.

In [ ]:
USE_VLLM = True

!pip -q install -r requirements.txt
if USE_VLLM:
    !pip -q install vllm
BACKEND = 'vllm' if USE_VLLM else 'hf'

## Unit tests for the sandbox and code extraction

In [ ]:
!python -m unittest discover tests

## Task 2: does the model load and write code?

In [ ]:
!python scripts/check_model.py --backend {BACKEND}

## Task 9: Run A

Greedy pass@1 on test and heldout, plus 8 samples per problem for pass@8 (finding 5).

In [ ]:
!python scripts/evaluate.py --run A --splits test heldout --pass-at-k 8 --backend {BACKEND} --output-dir {OUTPUT_DIR}

## Results

In [ ]:
import json, glob
for path in sorted(glob.glob(f'{OUTPUT_DIR}/A/*/*/summary.json')):
    s = json.load(open(path))
    extra = {k: round(v, 4) for k, v in s.items() if k.startswith('pass@') and k != 'pass@1'}
    print(f"{s['split']:8s} {s['mode']:8s} pass@1={s['pass@1']:.4f} {extra} "
          f"solved>=1: {s['problems_solved_at_least_once']}/{s['n_problems']}  "
          f"shown-pass-hidden-fail: {s['shown_pass_hidden_fail']}  ({s['seconds']}s)")
    print('          ', s['status_counts'])